# 03 — RCA Exploration

**Project:** AdIntel AI  
**Milestone:** 1 — Root Cause Exploration

Business case:

> Revenue remains stable, but viewability drops around 20% and video completion rate declines.

Notebook ini mengeksplorasi kemungkinan root cause dari sisi:

- Advertiser mix
- Campaign objective
- Placement quality
- Device/platform
- Market
- Inventory type
- Creative duration
- Data quality issue


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
ISSUE_START_DATE = pd.Timestamp("2026-03-15")

print("Data dir:", DATA_DIR)


Data dir: data\synthetic


## 1. Load Data


In [2]:
advertisers = pd.read_csv(DATA_DIR / "advertisers.csv")
campaigns = pd.read_csv(DATA_DIR / "campaigns.csv")
ad_groups = pd.read_csv(DATA_DIR / "ad_groups.csv")
creatives = pd.read_csv(DATA_DIR / "creatives.csv")
placements = pd.read_csv(DATA_DIR / "placements.csv")
markets = pd.read_csv(DATA_DIR / "markets.csv")
devices = pd.read_csv(DATA_DIR / "devices.csv")
perf = pd.read_csv(DATA_DIR / "daily_ad_performance.csv")
video = pd.read_csv(DATA_DIR / "video_performance.csv")
conversions = pd.read_csv(DATA_DIR / "conversion_events.csv")
billing = pd.read_csv(DATA_DIR / "billing_revenue.csv")
inventory = pd.read_csv(DATA_DIR / "inventory.csv")
dq_logs = pd.read_csv(DATA_DIR / "data_quality_logs.csv")

perf["date"] = pd.to_datetime(perf["date"])
video["date"] = pd.to_datetime(video["date"])
inventory["date"] = pd.to_datetime(inventory["date"])
dq_logs["date"] = pd.to_datetime(dq_logs["date"])

perf["period"] = np.where(perf["date"] < ISSUE_START_DATE, "pre", "post")
video["period"] = np.where(video["date"] < ISSUE_START_DATE, "pre", "post")
inventory["period"] = np.where(inventory["date"] < ISSUE_START_DATE, "pre", "post")
dq_logs["period"] = np.where(dq_logs["date"] < ISSUE_START_DATE, "pre", "post")

print("Performance rows:", len(perf))


FileNotFoundError: [Errno 2] No such file or directory: 'data\\synthetic\\advertisers.csv'

## 2. Build Enriched Performance Table


In [ ]:
perf_enriched = (
    perf
    .merge(advertisers, on="advertiser_id", how="left")
    .merge(campaigns[["campaign_id", "campaign_name", "objective", "buying_type", "daily_budget_usd"]], on="campaign_id", how="left")
    .merge(ad_groups[["ad_group_id", "targeting_type", "optimization_goal", "bid_strategy"]], on="ad_group_id", how="left")
    .merge(creatives[["creative_id", "creative_format", "is_video", "video_duration_sec", "creative_quality_score"]], on="creative_id", how="left")
    .merge(placements[["placement_id", "placement_name", "page_type", "placement_position", "inventory_type", "quality_tier", "is_below_the_fold"]], on="placement_id", how="left")
    .merge(markets[["market_id", "market_name", "region", "market_maturity"]], on="market_id", how="left")
    .merge(devices[["device_id", "device_type", "platform", "os_family", "is_app"]], on="device_id", how="left")
)

perf_enriched.head()


## 3. Helper Functions


In [ ]:
def safe_divide(num, den):
    return np.where(den == 0, np.nan, num / den)


def summarize_perf(df, group_cols):
    summary = (
        df.groupby(group_cols, dropna=False)
        .agg(
            impressions=("impressions", "sum"),
            clicks=("clicks", "sum"),
            spend_usd=("spend_usd", "sum"),
            revenue_usd=("publisher_revenue_usd", "sum"),
            measurable_impressions=("measurable_impressions", "sum"),
            viewable_impressions=("viewable_impressions", "sum"),
        )
        .reset_index()
    )
    summary["ctr"] = summary["clicks"] / summary["impressions"]
    summary["cpc"] = summary["spend_usd"] / summary["clicks"].replace(0, np.nan)
    summary["cpm"] = summary["spend_usd"] / summary["impressions"] * 1000
    summary["ecpm"] = summary["revenue_usd"] / summary["impressions"] * 1000
    summary["viewability"] = summary["viewable_impressions"] / summary["measurable_impressions"]
    return summary


def pre_post_movement(summary, dim_col, metric_cols):
    pre = summary[summary["period"] == "pre"][[dim_col] + metric_cols].copy()
    post = summary[summary["period"] == "post"][[dim_col] + metric_cols].copy()

    merged = pre.merge(post, on=dim_col, how="outer", suffixes=("_pre", "_post"))

    for metric in metric_cols:
        merged[f"{metric}_change"] = merged[f"{metric}_post"] / merged[f"{metric}_pre"] - 1

    return merged


def weighted_avg(df, value_col, weight_col):
    weight = df[weight_col].sum()
    if weight == 0:
        return np.nan
    return (df[value_col] * df[weight_col]).sum() / weight


## 4. Executive Summary — Overall Pre vs Post


In [ ]:
overall = summarize_perf(perf_enriched, ["period"])
overall


In [ ]:
overall_movement = pd.DataFrame({
    "metric": ["impressions", "revenue_usd", "spend_usd", "cpm", "ecpm", "viewability"],
    "pre": [
        overall.loc[overall["period"] == "pre", "impressions"].iloc[0],
        overall.loc[overall["period"] == "pre", "revenue_usd"].iloc[0],
        overall.loc[overall["period"] == "pre", "spend_usd"].iloc[0],
        overall.loc[overall["period"] == "pre", "cpm"].iloc[0],
        overall.loc[overall["period"] == "pre", "ecpm"].iloc[0],
        overall.loc[overall["period"] == "pre", "viewability"].iloc[0],
    ],
    "post": [
        overall.loc[overall["period"] == "post", "impressions"].iloc[0],
        overall.loc[overall["period"] == "post", "revenue_usd"].iloc[0],
        overall.loc[overall["period"] == "post", "spend_usd"].iloc[0],
        overall.loc[overall["period"] == "post", "cpm"].iloc[0],
        overall.loc[overall["period"] == "post", "ecpm"].iloc[0],
        overall.loc[overall["period"] == "post", "viewability"].iloc[0],
    ],
})

overall_movement["change"] = overall_movement["post"] / overall_movement["pre"] - 1
overall_movement["change_pct"] = overall_movement["change"].map(lambda x: f"{x:.2%}")
overall_movement


## 5. Trend: Revenue, Impression, Viewability


In [ ]:
daily = summarize_perf(perf_enriched, ["date"])

plt.figure(figsize=(12, 5))
plt.plot(daily["date"], daily["revenue_usd"])
plt.axvline(ISSUE_START_DATE, linestyle="--")
plt.title("Daily Revenue Trend")
plt.xlabel("Date")
plt.ylabel("Revenue USD")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(daily["date"], daily["impressions"])
plt.axvline(ISSUE_START_DATE, linestyle="--")
plt.title("Daily Impression Trend")
plt.xlabel("Date")
plt.ylabel("Impressions")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(daily["date"], daily["viewability"])
plt.axvline(ISSUE_START_DATE, linestyle="--")
plt.title("Daily Viewability Trend")
plt.xlabel("Date")
plt.ylabel("Viewability")
plt.xticks(rotation=45)
plt.show()


## 6. RCA 1 — Placement Quality Mix


In [ ]:
quality_summary = summarize_perf(perf_enriched, ["period", "quality_tier"])
quality_summary["impression_share"] = quality_summary.groupby("period")["impressions"].transform(lambda x: x / x.sum())
quality_summary.sort_values(["period", "quality_tier"])


In [ ]:
quality_movement = pre_post_movement(
    quality_summary,
    "quality_tier",
    ["impressions", "impression_share", "viewability", "ecpm", "revenue_usd"]
)

quality_movement.sort_values("impression_share_change", ascending=False)


In [ ]:
quality_pivot = quality_summary.pivot(index="quality_tier", columns="period", values="impression_share")
quality_pivot.plot(kind="bar", figsize=(9, 5))
plt.title("Impression Share by Placement Quality Tier")
plt.xlabel("Quality Tier")
plt.ylabel("Impression Share")
plt.xticks(rotation=0)
plt.show()


**Interpretation guide:**

Kalau `Low` quality impression share naik post-issue dan viewability overall turun, ini kandidat root cause utama.


## 7. RCA 2 — Device / Platform


In [ ]:
platform_summary = summarize_perf(perf_enriched, ["period", "platform"])
platform_summary["impression_share"] = platform_summary.groupby("period")["impressions"].transform(lambda x: x / x.sum())
platform_summary.sort_values(["period", "platform"])


In [ ]:
platform_movement = pre_post_movement(
    platform_summary,
    "platform",
    ["impressions", "impression_share", "viewability", "ecpm", "revenue_usd"]
)

platform_movement.sort_values("viewability_change")


In [ ]:
platform_view = platform_summary.pivot(index="platform", columns="period", values="viewability")
platform_view.plot(kind="bar", figsize=(9, 5))
plt.title("Viewability by Platform")
plt.xlabel("Platform")
plt.ylabel("Viewability")
plt.xticks(rotation=0)
plt.show()


**Interpretation guide:**

Jika mobile web share naik dan mobile web viewability lebih rendah dibanding app, maka device/platform adalah root cause kuat.


## 8. RCA 3 — Placement Quality x Platform


In [ ]:
quality_platform = summarize_perf(perf_enriched, ["period", "quality_tier", "platform"])
quality_platform["impression_share"] = quality_platform.groupby("period")["impressions"].transform(lambda x: x / x.sum())

quality_platform.sort_values(["period", "impression_share"], ascending=[True, False]).head(20)


In [ ]:
post_problem_segments = (
    quality_platform[quality_platform["period"] == "post"]
    .sort_values(["impression_share", "viewability"], ascending=[False, True])
    .head(10)
)

post_problem_segments


## 9. RCA 4 — Inventory Type


In [ ]:
inventory_type_summary = summarize_perf(perf_enriched, ["period", "inventory_type"])
inventory_type_summary["impression_share"] = inventory_type_summary.groupby("period")["impressions"].transform(lambda x: x / x.sum())

inventory_type_summary


In [ ]:
inventory_type_movement = pre_post_movement(
    inventory_type_summary,
    "inventory_type",
    ["impressions", "impression_share", "viewability", "ecpm", "revenue_usd"]
)

inventory_type_movement.sort_values("impression_share_change", ascending=False)


## 10. RCA 5 — Creative Duration and VCR


In [ ]:
video_enriched = (
    video
    .merge(perf_enriched[[
        "performance_id",
        "advertiser_id",
        "campaign_id",
        "placement_id",
        "device_id",
        "market_id",
        "quality_tier",
        "platform",
        "inventory_type",
        "objective",
        "creative_id",
        "video_duration_sec",
        "creative_quality_score",
    ]], on=["performance_id", "creative_id"], how="left")
)

video_enriched["duration_bucket"] = pd.cut(
    video_enriched["video_duration_sec"],
    bins=[0, 10, 15, 30, 60],
    labels=["<=10s", "11-15s", "16-30s", "31-60s"],
    include_lowest=True
)

video_summary = (
    video_enriched.groupby(["period", "duration_bucket"], observed=False)
    .agg(
        video_starts=("video_starts", "sum"),
        video_completes=("video_completes", "sum"),
    )
    .reset_index()
)
video_summary["vcr"] = video_summary["video_completes"] / video_summary["video_starts"]
video_summary["start_share"] = video_summary.groupby("period")["video_starts"].transform(lambda x: x / x.sum())

video_summary


In [ ]:
duration_movement = pre_post_movement(
    video_summary,
    "duration_bucket",
    ["video_starts", "start_share", "vcr"]
)

duration_movement.sort_values("start_share_change", ascending=False)


In [ ]:
vcr_pivot = video_summary.pivot(index="duration_bucket", columns="period", values="vcr")
vcr_pivot.plot(kind="bar", figsize=(9, 5))
plt.title("VCR by Creative Duration Bucket")
plt.xlabel("Duration Bucket")
plt.ylabel("VCR")
plt.xticks(rotation=0)
plt.show()


**Interpretation guide:**

Jika long video bucket naik share-nya dan VCR-nya lebih rendah, creative duration menjadi partial root cause.


## 11. RCA 6 — Advertiser Mix


In [ ]:
advertiser_summary = summarize_perf(perf_enriched, ["period", "advertiser_id", "advertiser_name", "industry", "advertiser_tier"])
advertiser_summary["impression_share"] = advertiser_summary.groupby("period")["impressions"].transform(lambda x: x / x.sum())
advertiser_summary["revenue_share"] = advertiser_summary.groupby("period")["revenue_usd"].transform(lambda x: x / x.sum())

advertiser_post = advertiser_summary[advertiser_summary["period"] == "post"].sort_values("impressions", ascending=False).head(15)
advertiser_post


In [ ]:
tier_summary = summarize_perf(perf_enriched, ["period", "advertiser_tier"])
tier_summary["impression_share"] = tier_summary.groupby("period")["impressions"].transform(lambda x: x / x.sum())
tier_summary["revenue_share"] = tier_summary.groupby("period")["revenue_usd"].transform(lambda x: x / x.sum())

tier_summary


In [ ]:
tier_movement = pre_post_movement(
    tier_summary,
    "advertiser_tier",
    ["impressions", "impression_share", "revenue_usd", "revenue_share", "viewability", "ecpm"]
)

tier_movement.sort_values("impression_share_change", ascending=False)


## 12. RCA 7 — Campaign Objective


In [ ]:
objective_summary = summarize_perf(perf_enriched, ["period", "objective"])
objective_summary["impression_share"] = objective_summary.groupby("period")["impressions"].transform(lambda x: x / x.sum())
objective_summary["revenue_share"] = objective_summary.groupby("period")["revenue_usd"].transform(lambda x: x / x.sum())

objective_summary


In [ ]:
objective_movement = pre_post_movement(
    objective_summary,
    "objective",
    ["impressions", "impression_share", "revenue_usd", "revenue_share", "viewability", "ecpm"]
)

objective_movement.sort_values("impression_share_change", ascending=False)


## 13. RCA 8 — Market


In [ ]:
market_summary = summarize_perf(perf_enriched, ["period", "market_name", "region", "market_maturity"])
market_summary["impression_share"] = market_summary.groupby("period")["impressions"].transform(lambda x: x / x.sum())
market_summary["revenue_share"] = market_summary.groupby("period")["revenue_usd"].transform(lambda x: x / x.sum())

market_summary.sort_values(["period", "impressions"], ascending=[True, False])


In [ ]:
market_movement = pre_post_movement(
    market_summary,
    "market_name",
    ["impressions", "impression_share", "revenue_usd", "revenue_share", "viewability", "ecpm"]
)

market_movement.sort_values("viewability_change")


## 14. Inventory Supply Side Check


In [ ]:
inventory_enriched = (
    inventory
    .merge(placements[["placement_id", "quality_tier", "inventory_type"]], on="placement_id", how="left")
    .merge(devices[["device_id", "platform"]], on="device_id", how="left")
    .merge(markets[["market_id", "market_name", "region"]], on="market_id", how="left")
)

inventory_summary = (
    inventory_enriched.groupby(["period"], as_index=False)
    .agg(
        ad_requests=("ad_requests", "sum"),
        eligible_requests=("eligible_requests", "sum"),
        bid_requests=("bid_requests", "sum"),
        bid_responses=("bid_responses", "sum"),
        won_impressions=("won_impressions", "sum"),
        avg_inventory_quality_score=("inventory_quality_score", "mean"),
    )
)
inventory_summary["fill_rate"] = inventory_summary["won_impressions"] / inventory_summary["ad_requests"]
inventory_summary["win_rate"] = inventory_summary["won_impressions"] / inventory_summary["bid_requests"]

inventory_summary


In [ ]:
inv_quality_summary = (
    inventory_enriched.groupby(["period", "quality_tier", "platform"], as_index=False)
    .agg(
        ad_requests=("ad_requests", "sum"),
        won_impressions=("won_impressions", "sum"),
        avg_inventory_quality_score=("inventory_quality_score", "mean"),
    )
)
inv_quality_summary["request_share"] = inv_quality_summary.groupby("period")["ad_requests"].transform(lambda x: x / x.sum())

inv_quality_summary.sort_values(["period", "request_share"], ascending=[True, False]).head(20)


## 15. Data Quality Check


In [ ]:
dq_logs.sort_values("date")


In [ ]:
dq_summary = (
    dq_logs.groupby(["period", "issue_type", "severity"], as_index=False)
    .agg(
        logs=("dq_log_id", "count"),
        estimated_affected_rows=("estimated_affected_rows", "sum"),
        root_cause_candidates=("is_root_cause_candidate", "sum"),
    )
)

dq_summary


In [ ]:
total_perf_rows = len(perf)
dq_affected_rows = dq_logs["estimated_affected_rows"].sum()
dq_affected_ratio = dq_affected_rows / total_perf_rows

print(f"Total performance rows       : {total_perf_rows:,}")
print(f"Estimated DQ affected rows   : {dq_affected_rows:,}")
print(f"Estimated affected row ratio : {dq_affected_ratio:.2%}")


**Interpretation guide:**

Jika estimated affected row ratio kecil dan severity mayoritas Low/Medium, maka data quality issue kemungkinan bukan root cause utama.


## 16. RCA Summary Table


In [ ]:
# Pull key evidence dynamically from previous summaries

low_quality_pre = quality_summary[(quality_summary["period"] == "pre") & (quality_summary["quality_tier"] == "Low")]["impression_share"].sum()
low_quality_post = quality_summary[(quality_summary["period"] == "post") & (quality_summary["quality_tier"] == "Low")]["impression_share"].sum()

mobile_web_pre = platform_summary[(platform_summary["period"] == "pre") & (platform_summary["platform"] == "Mobile Web")]["impression_share"].sum()
mobile_web_post = platform_summary[(platform_summary["period"] == "post") & (platform_summary["platform"] == "Mobile Web")]["impression_share"].sum()

viewability_change = overall_movement.loc[overall_movement["metric"] == "viewability", "change"].iloc[0]
revenue_change = overall_movement.loc[overall_movement["metric"] == "revenue_usd", "change"].iloc[0]
impression_change = overall_movement.loc[overall_movement["metric"] == "impressions", "change"].iloc[0]

long_video_pre = video_summary[(video_summary["period"] == "pre") & (video_summary["duration_bucket"].isin(["16-30s", "31-60s"]))]["start_share"].sum()
long_video_post = video_summary[(video_summary["period"] == "post") & (video_summary["duration_bucket"].isin(["16-30s", "31-60s"]))]["start_share"].sum()

rca_summary = pd.DataFrame([
    {
        "rca_dimension": "Placement Quality",
        "finding": "Low-quality placement impression share increased post issue",
        "evidence": f"Low-quality share: {low_quality_pre:.2%} -> {low_quality_post:.2%}",
        "root_cause_likelihood": "High"
    },
    {
        "rca_dimension": "Device / Platform",
        "finding": "Mobile web impression share increased and has weaker viewability",
        "evidence": f"Mobile web share: {mobile_web_pre:.2%} -> {mobile_web_post:.2%}",
        "root_cause_likelihood": "High"
    },
    {
        "rca_dimension": "Creative Duration",
        "finding": "Longer video duration contributes to lower VCR",
        "evidence": f"Long video start share: {long_video_pre:.2%} -> {long_video_post:.2%}",
        "root_cause_likelihood": "Medium"
    },
    {
        "rca_dimension": "Revenue Dynamics",
        "finding": "Revenue remains stable because impression volume offsets lower quality/eCPM",
        "evidence": f"Revenue change: {revenue_change:.2%}; Impression change: {impression_change:.2%}",
        "root_cause_likelihood": "Explains business symptom"
    },
    {
        "rca_dimension": "Data Quality",
        "finding": "DQ issues exist but are limited and localized",
        "evidence": f"Estimated affected row ratio: {dq_affected_ratio:.2%}",
        "root_cause_likelihood": "Low"
    },
])

rca_summary


## 17. Draft Business Conclusion

Gunakan conclusion ini sebagai starting point untuk dokumentasi portfolio.


In [ ]:
conclusion = f'''
Revenue stayed broadly stable post issue because impression volume increased materially, offsetting lower quality and softer eCPM.
However, viewability declined by {viewability_change:.2%}, indicating a quality deterioration rather than a pure monetization issue.

The strongest root cause candidates are:
1. Placement quality mix shift: low-quality placement share increased from {low_quality_pre:.2%} to {low_quality_post:.2%}.
2. Device/platform mix shift: mobile web share increased from {mobile_web_pre:.2%} to {mobile_web_post:.2%}.
3. Creative duration effect: longer video creatives contributed to weaker video completion rate.

Data quality issues were detected, but the estimated affected volume was only {dq_affected_ratio:.2%} of performance rows,
so tracking issue is unlikely to be the primary root cause.
'''

print(conclusion)


## 18. Recommended Next Actions

Suggested business actions:

1. Reduce allocation to low-quality mobile web placements.
2. Introduce placement quality guardrail for campaigns optimized for reach/video views.
3. Split reporting for App vs Mobile Web quality metrics.
4. Encourage shorter video creatives for mobile web inventory.
5. Monitor data quality logs, but do not treat DQ as the main root cause unless issue volume increases.
